In [1]:
# Helper notebook that streamlines data preparation for analysis by
# converting CSV files into FITS format for faster processing.
# The workflow first generates CSV files using the `make_csv_files`
# function (documented in a separate notebook), merges them into a
# single master dataset, and converts the final master file to FITS.

import pandas as pd
from datetime import datetime, timedelta
import requests
import glob
from astropy.io import fits
%run make_csv_files.ipynb

In [52]:
# Use function to create all csv files between Jan 1 - Dec 31 in specified year
make_csv_files('choose_year_between_2009-current')

               datetime          500m           1km           2km  \
0   2017-01-03 19:18:31  9.380000e-22  2.610000e-23  1.920000e-22   
1   2017-01-03 19:20:01  4.400000e-22  1.570000e-21  1.330000e-20   
2   2017-01-03 19:21:32  1.130000e-22  5.250000e-22  4.960000e-21   
3   2017-01-03 19:24:29  1.240000e-25  1.130000e-22  3.080000e-21   
4   2017-01-03 19:25:58  3.030000e-23  8.020000e-24  5.710000e-21   
..                  ...           ...           ...           ...   
99  2017-01-04 03:15:00  2.720000e-23  3.020000e-22  4.010000e-20   
100 2017-01-04 03:19:33  4.580000e-22  8.890000e-23  6.820000e-20   
101 2017-01-04 04:53:03  4.910000e-23  2.090000e-22  5.000000e-21   
102 2017-01-04 05:22:06  1.480000e-21  8.240000e-24  1.570000e-14   
103 2017-01-04 05:32:47  2.090000e-21  1.720000e-21  1.140000e-20   

              4km           8km          16km  mass_val  dimm_val  \
0    1.070000e-16  2.180000e-14  1.460000e-14      0.17      0.34   
1    1.550000e-14  4.950000e-14  

In [33]:
# Combine '{date}_results.csv' to one data file labeled '{year}.csv'
# Match all CSV files in a folder
csv_files = glob.glob("*_result.csv")

# Read CSVs, remove empty DataFrames and all-NA columns
df_list = []
for file in csv_files:
    df = pd.read_csv(file)
    if not df.empty:  # skip empty DataFrames
        df = df.dropna(axis=1, how="all")  # drop columns that are all NA
        df_list.append(df)

# Merge into one DataFrame
merged_df = pd.concat(df_list, ignore_index=True)

# Sort by the first column
first_column_name = merged_df.columns[0]
merged_df = merged_df.sort_values(by=first_column_name)

# Save the merged file
merged_df.to_csv("2025_900sec.csv", index=False)

In [35]:
# Convert all {year}.csv files to one master file 
file = './years_tol900sec/*.csv'
year_csv_files = glob.glob(file)

# Read and concatenate all CSVs
df_list = [pd.read_csv(file) for file in year_csv_files]
combined_df = pd.concat(df_list, ignore_index=True)

# Save to a new CSV
combined_df.to_csv("master_file_900sec.csv", index=False)

In [3]:
# Convert master_file.csv to master_file.fits
from astropy.table import Table

# Read CSV
csv_file = "revised.csv"
df = pd.read_csv(csv_file)

# Convert to Astropy Table
table = Table.from_pandas(df)

# Save to FITS
fits_file = "revised.fits"
table.write(fits_file, format='fits', overwrite=True)

In [ ]:
# Test if conversion was successful
# Open the FITS file
fits_file = "master_file.fits"
hdul = fits.open(fits_file)

# Print summary of the file
hdul.info()

# Print header of a table HDU (e.g., first extension)
print(hdul[1].header)

# Access data in the first extension
data = hdul[1].data  # or hdul[0].data for primary HDU
print(data)

# List column names if it's a table
print(data.columns.names)